# The Brain of AI agents: LLM

Here, we
- Select the right LLM for agent development
- Use LiteLLM to model against a common LLM API facade
- Engineer prompts for agents
- Setup GAIA benchmark and make LLMs limitations obvious

In [1]:
from ollama import chat
from ollama import ChatResponse

response: ChatResponse = chat(
  model="qwen3",
  messages=[
    {"role": "system", "content": "You are a helpful assistant, providing concise answers."},
    {"role": "user", "content": "Why is the sky blue?"}
  ],
    stream=True,
)

for chunk in response:
  print(chunk.message.content, end="", flush=True)

The sky appears blue due to **Rayleigh scattering**, where shorter wavelengths of light (like blue and violet) scatter more effectively in Earth's atmosphere. While violet scatters the most, the sun emits more blue light, and our eyes are more sensitive to blue. Additionally, some violet light is absorbed by the upper atmosphere. During sunrise/sunset, longer paths through the atmosphere scatter out blue light, leaving reds and oranges.

## Nutze LiteLLM

Wir nutzen nun LiteLLM um gegen eine einheitliche API zu modellieren.

In [2]:
from litellm import completion

response = completion(
  model="ollama/qwen3", 
  messages=[{"role": "user", "content": "Why is the sky blue?"}],
)

print(response.choices[0].message.content)

The sky appears blue due to a phenomenon called **Rayleigh scattering**, which involves how sunlight interacts with the Earth's atmosphere. Here's a breakdown of the process:

1. **Sunlight Composition**: Sunlight is a mix of all visible colors (red, orange, yellow, green, blue, indigo, violet), each with different wavelengths. Blue and violet have the shortest wavelengths, while red and yellow have longer ones.

2. **Scattering in the Atmosphere**: When sunlight enters Earth's atmosphere, it collides with molecules and small particles (like nitrogen and oxygen). These interactions cause the light to scatter in all directions. **Shorter wavelengths (blue/violet) scatter more** than longer wavelengths (red/yellow), a process described by Rayleigh scattering.

3. **Why Blue, Not Violet?**: While violet light scatters the most, our eyes are more sensitive to blue, and the Sun emits more blue light than violet. Additionally, some violet light is absorbed by the upper atmosphere, making blu

## Statusmanagement mit APIs

Wir müssen uns den Verlauf der Konversation explizit merken. Ansonsten fangen wir mit jedem API-Call bei 

In [3]:
from litellm import completion

# First call
response1 = completion(
    model="ollama/qwen3",
    messages=[{"role": "user", "content": "My name is Thorsten."}]
)
print(response1.choices[0].message.content)
 
# Second call
response2 = completion(
    model="ollama/qwen3",
    messages=[{"role": "user", "content": "What is my name?"}]
)
print(response2.choices[0].message.content)

Hello, Thorsten! How can I assist you today? 😊
I don’t have access to personal information about you, including your name. If you have any questions or need assistance, feel free to let me know! 😊


Nun merken wir uns explizit den Verlauf unserer Konversation, und teilen sie dem LLM mit:

In [4]:
from litellm import completion

messages = []

# First call
messages.append({"role": "user", "content": "My name is Thorsten."})

response1 = completion(model="ollama/qwen3", messages=messages)
assistant_message1 = response1.choices[0].message.content;
messages.append({"role": "assistant", "content": assistant_message1})
print(assistant_message1)
 
# Second call
messages.append({"role": "user", "content": "What is my name?"})
response1 = completion(model="ollama/qwen3", messages=messages)
assistant_message1 = response1.choices[0].message.content;
messages.append({"role": "assistant", "content": assistant_message1})
print(assistant_message1)

Hello, Thorsten! Nice to meet you. How can I assist you today? 😊
Your name is Thorsten! 😊 Is there anything I can help you with today?


## Generiere strukturiere Ausgaben mit Pydantic

Damit wir die Ausgaben durch Tools verarbeiten können, möchten wir Antworten gemäß einem definierten JSON-Schema erzeugen.

In [5]:
from pydantic import BaseModel
from litellm import completion
 
class ExtractedInfo(BaseModel):
    name: str
    email: str
    phone: str | None = None
 
response = completion(
    model="ollama/qwen3",
    messages=[{
        "role": "user", 
        "content": "My name is John Smith, my email is john@example.com, and my phone is 555-1234."
    }],
    response_format=ExtractedInfo
)
 
result = response.choices[0].message.content

print(result)

{"name": "John Smith", "email": "john@example.com", "phone": "555-1234"}



## Make asynchronous LLM calls

In [6]:
import asyncio
from litellm import acompletion
import warnings

warnings.filterwarnings('ignore', category=UserWarning, module='pydantic')

async def get_response(prompt: str) -> str:
    response = await acompletion(
        model="ollama/qwen3",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

prompts = [
    "What is 2 + 2?",
    "What is the capital of Germany?",
    "Who wrote Romeo and Juliet?"
]

tasks = [get_response(p) for p in prompts]
results = await asyncio.gather(*tasks)

for prompt, answer in zip(prompts, results):
    print(f"Q: {prompt}")
    print(f"A: {answer}\n")

Q: What is 2 + 2?
A: The result of 2 + 2 is **4**.

Q: What is the capital of Germany?
A: The capital of Germany is **Berlin**. It is also the largest city in the country and serves as the seat of the federal government. Berlin has been the capital since the unification of Germany in 1871 and remains the political, cultural, and economic center of the nation.

Q: Who wrote Romeo and Juliet?
A: "Romeo and Juliet" is widely attributed to **William Shakespeare**, an English playwright and poet from the late 16th century. The play was first published in 1597 as part of the "Quarto" edition and later included in the First Folio of 1623, which is the first collected edition of Shakespeare's works.

While Shakespeare is universally recognized as the author, there have been historical debates and alternative theories about his authorship (e.g., claims that Francis Bacon or Edward de Vere, 17th Earl of Oxford, wrote it). However, these theories lack substantial evidence, and the overwhelming co

In [ ]:
semaphore = asyncio.Semaphore(2)

async def call_llm(prompt: str) -> str:
    """Call the LLM with automatic retry and a semaphore to limit concurrency."""
    async with semaphore:
        response = await acompletion(
            model="ollama/gpt-oss:120b-cloud",
            messages=[{"role": "user", "content": prompt}],
            num_retries=3
        )
        return response.choices[0].message.content

# Now, only 10 API calls run cuncurrently
prompts = [f"What is {i} + {i}?" for i in range(100)]
tasks = [call_llm(p) for p in prompts]

results  = await asyncio.gather(*tasks, return_exceptions=True)

for prompt, answer in zip(prompts, results):
    print(f"Q: {prompt}")
    print(f"A: {answer}\n")


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.I